# 反向传播机制

## 矩阵求导
- 深度学习里面采用的分母布局——求导结果的0维等于分母（即x的维度），1维等于分子（即y的维度）（具体可以看下面的公式）
- 分子布局和分母布局结果一样，就只是相差一个转置

### 标量对向量求导
- 因变量是个标量，但自变量是向量

- $W_{n\times 1}, x_{n\times 1}$
- $f(x) = W^Tx$
- $\frac{\partial (\mathbf{w}^T \mathbf{x})}{\partial \mathbf{x}} = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix} = \begin{bmatrix} w_1 \\ w_2 \\ \vdots \\ w_n \end{bmatrix} = \mathbf{w}$

- $f(\mathbf{x}) = \mathbf{x}^T \mathbf{A} \mathbf{x}$
- $\frac{\partial (\mathbf{x}^T \mathbf{A} \mathbf{x})}{\partial \mathbf{x}} = \mathbf{A} \mathbf{x} + \mathbf{A}^T \mathbf{x}$

### 向量对标量求导
- 因变量是向量，自变量是标量

- $\mathbf{y} = \begin{bmatrix} t^2 \\ \sin(t) \\ 3t \end{bmatrix}$

- $\frac{\partial \mathbf{y}}{\partial t} = \begin{bmatrix} \frac{\partial (t^2)}{\partial t} & \frac{\partial (\sin(t))}{\partial t} & \frac{\partial (3t)}{\partial t} \end{bmatrix} = \begin{bmatrix} 2t & \cos(t) & 3 \end{bmatrix}$

### 向量对向量求导——雅可比矩阵

- $\frac{\partial \mathbf{y}}{\partial \mathbf{x}} = \begin{bmatrix} \frac{\partial y_1}{\partial x_1} & \frac{\partial y_2}{\partial x_1} & \dots & \frac{\partial y_m}{\partial x_1} \\ \frac{\partial y_1}{\partial x_2} & \frac{\partial y_2}{\partial x_2} & \dots & \frac{\partial y_m}{\partial x_2} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial y_1}{\partial x_n} & \frac{\partial y_2}{\partial x_n} & \dots & \frac{\partial y_m}{\partial x_n} \end{bmatrix}$

## 反向传播机制
以一个简单的神经网络为例，描述其数学原理

- 第一层线性变换：$\mathbf{z} = \mathbf{W}\mathbf{x}$ （$\mathbf{z}$ 是向量）——$\frac{\partial \mathbf{z}}{\partial \mathbf{x}}$
- 激活函数：$\mathbf{a} = \text{ReLU}(\mathbf{z})$ （$\mathbf{a}$ 是向量）——$\frac{\partial \mathbf{a}}{\partial \mathbf{z}}$
- 计算损失：$L = \text{MSE}(\mathbf{a}, \mathbf{y}_{true})$ （$L$ 是标量）——$\frac{\partial L}{\partial \mathbf{a}}$



- **_则链式求导结果是——$\frac{\partial L}{\partial \mathbf{x}} = \frac{\partial L}{\partial \mathbf{a}}  \times \frac{\partial \mathbf{a}}{\partial \mathbf{z}} \times \frac{\partial \mathbf{z}}{\partial \mathbf{x}}$（注意这里的运算顺序是反向的）_**
- 反向传播——先计算损失函数的梯度，然后依次反向经过隐藏层，激活函数，输入函数，最后优化参数

In [3]:
import math


class Value:
    def __init__(self, data, inputs=()):
        # 节点的值，前向计算时算出
        self.data = data
        # 节点的梯度，即损失对这个节点的导数，反向传播时才算出来
        # 初始为 0，节点被多处用到时，每条路径传回的梯度都加到这里
        self.grad = 0.0
        # 算出这个节点用到的节点，比如 u = w * x 的 inputs 是 (w, x)
        # 反向传播时顺着它往回走
        self.inputs = inputs
        # 把梯度传给输入：用自己的 grad 乘上局部导数，加到每个输入的 grad 上
        # 由创建这个节点的运算设置；w、b 这样直接创建的节点没有输入，什么也不做
        self.pass_grad = lambda: None

    def __add__(self, other):
        # 普通数字先包装成 Value
        other = other if isinstance(other, Value) else Value(other)
        # 前向计算：算出结果，同时记下两个输入，计算图就是这样一步步记录下来的
        out = Value(self.data + other.data, (self, other))

        # 加法的局部导数是 1，out 的梯度原样传给两个输入
        def pass_grad():
            self.grad += out.grad
            other.grad += out.grad

        # 先存起来，等反向传播走到 out 时才调用
        out.pass_grad = pass_grad
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        # a·b 对 a 的局部导数是 b，对 b 的局部导数是 a
        # 所以每个输入得到的梯度，等于另一个输入的值乘上 out 的梯度
        def pass_grad():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out.pass_grad = pass_grad
        return out

    def relu(self):
        out = Value(max(self.data, 0.0), (self,))

        # ReLU 的局部导数：输入大于 0 时是 1，梯度原样通过；否则是 0，不传梯度
        def pass_grad():
            if self.data > 0:
                self.grad += out.grad

        out.pass_grad = pass_grad
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        # e^x 的导数还是 e^x，也就是前向算出的 out.data
        def pass_grad():
            self.grad += out.data * out.grad

        out.pass_grad = pass_grad
        return out

    def log(self):
        out = Value(math.log(self.data), (self,))

        # log x 的导数是 1/x
        def pass_grad():
            self.grad += out.grad / self.data

        out.pass_grad = pass_grad
        return out

    # 取负就是乘以 -1，减法就是加上取负后的数
    # 求导交给已有的乘法和加法，不用单独写
    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other)

    def backward(self):
        # 第一步：把计算图中的节点按拓扑顺序排进 postorder，每个节点都排在它的输入节点后面
        postorder = []
        visited = set()

        # DFS 后序遍历：先排好所有输入节点，再把自己加进去
        # visited 保证被多处用到的节点只排一次
        def traverse(node):
            if node not in visited:
                visited.add(node)
                for child in node.inputs:
                    traverse(child)
                postorder.append(node)

        traverse(self)
        # 第二步：损失对自己的导数是 1，从损失出发倒着遍历
        # 轮到一个节点时，用到它的节点都已经处理完，它的梯度已经完整，可以继续传给输入
        self.grad = 1.0
        for node in reversed(postorder):
            node.pass_grad()

In [7]:
# 5 条配送记录：距离 x（公里）和实际用时 y（分钟）
xs = [1, 2, 3, 4, 5]
ys = [8, 12, 14, 17, 23]

w = Value(0.0)
b = Value(0.0)

# 前向计算均方误差：每做一次加、减、乘都会新建一个节点，算完 loss，计算图也就记录好了
loss = Value(0.0)
for x, y in zip(xs, ys):
    # w 和 b 在 5 条记录里都要用到，5 条路径的梯度会累加到它们的 grad 上
    error = w * x + b - y
    # 平方写成 error * error，error 是乘法的两个输入，收到的两份梯度加起来，正好对应平方的导数 2e
    loss = loss + error * error
loss = loss * (1 / len(xs))

# 反向传播：从 loss 倒着走一遍，w 和 b 的梯度一次都算出来
loss.backward()
print(f"loss = {loss.data:.4f}")
print(f"dw = {w.grad:.4f}, db = {b.grad:.4f}")

loss = 244.4000
dw = -102.8000, db = -29.6000


In [8]:
import torch

# 用 PyTorch 的自动求导算一遍同样的梯度
# requires_grad=True 表示要对这个张量求梯度，PyTorch 会在前向计算时记录计算图
w_t = torch.tensor(0.0, requires_grad=True)
b_t = torch.tensor(0.0, requires_grad=True)
x_t = torch.tensor(xs, dtype=torch.float32)
y_t = torch.tensor(ys, dtype=torch.float32)
loss_t = ((w_t * x_t + b_t - y_t) ** 2).mean()
# 和上面一样调用 backward()，梯度存在各自的 .grad 里
loss_t.backward()
print(f"loss = {loss_t.item():.4f}")
print(f"dw = {w_t.grad.item():.4f}, db = {b_t.grad.item():.4f}")

loss = 244.4000
dw = -102.8000, db = -29.6000
